# Creating clusters based on distance and time
This is the analysis part of our project where we will work with the data we aggregated before. We will first apply our clustering method to recive groups of fires and later convert them to an area.

Geometry clarification: From now on, fires are treated as point features. This approach is chosen because points are easier to handle, especially for the combined spatio-temporal analysis.

These points represent the centroid of the pixels provided by the VIIRS sensors. Since the edge length of these pixels is known, it would theoretically always be possible to reconstruct the original pixel geometry.

However, because there is no additional information about the exact location of fire activity within a pixel, storing polygons instead of points does not improve data quality or spatial precision. Therefore, this transformation is considered to have no negative impact on the analysis.

In [16]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree
import networkx as nx
from pathlib import Path

# -----------------------------
# LOAD DATA
# -----------------------------
# Ordnerpfad
processed_dir = Path("../data/processed")

# Nur CSV-Dateien mit "firms" im Namen
csv_files = processed_dir.glob("*firms*.csv")

# Dateien einlesen und kombinieren
df = pd.concat(
    [pd.read_csv(file) for file in csv_files],
    ignore_index=True
)

print(df.head())

       lat       lon   frp  time_sin  time_cos  date_sin  date_cos
0 -8.07199 -34.97615  0.96  0.798636  0.601815   0.54524  -0.83828
1 -7.00310 -43.86724  1.82  0.798636  0.601815   0.54524  -0.83828
2 -7.00029 -43.86731  4.51  0.798636  0.601815   0.54524  -0.83828
3 -6.99685 -43.86600  1.82  0.798636  0.601815   0.54524  -0.83828
4 -6.99594 -43.87103  2.27  0.798636  0.601815   0.54524  -0.83828


# Parameter selection
Select appropriate parameter for maximum distance between detected fires which belong to the same cluster and a time threshold which limits the time passed since the last detection of a fire (ensures a wildfire is still active).

In [17]:
# -----------------------------
# MAXIMUM DISTANCE in km between two points to be considered in the same cluster.
MAX_DISTANCE_KM = 1.5

# -----------------------------
# TIME_THRESHOLD
# Threshold based on the time difference between two events to be considered in the same cluster.
# Calculated with the time_dist function, which computes the distance between two time feature vectors.
# Values: Range from 0 to 2. A lower value means that events must be closer in time to be clustered together.
TIME_THRESHOLD = 0.8

# Applying clustering method based on distance and time

In [18]:
# -----------------------------
# SPATIAL + TIME FEATURES 
# -----------------------------

coords = np.radians(df[["lat", "lon"]].values) # Convert lat/lon to radians for haversine distance

# Time features: Convert time and date to cyclical features (sin and cos) 
# This prevents issues with the discontinuity of time (e.g., 23:59 and 00:00 are close in time)
time_features = df[[
    "time_sin", "time_cos",
    "date_sin", "date_cos"
]].values

In [19]:
# -----------------------------
# SPATIAL INDEX
# -----------------------------
# Efficient nearest neighbor method for spherical data which uses the Haversine distance metric
tree = BallTree(coords, metric="haversine") 

radius_km = MAX_DISTANCE_KM
radius = radius_km / 6371.0  # Earth radius in km


In [20]:
# -----------------------------
# DISTANCE FUNCTION (CYCLIC TIME)
# -----------------------------
# Computes the distance between two time feature vectors using Euclidean distance.
def time_dist(i, j):
    return np.linalg.norm(time_features[i] - time_features[j])


# -----------------------------
# GRAPH BUILD (Space-Time Cube)
# -----------------------------
# Build a graph where nodes are events and edges connect events that are close in space and time.
G = nx.Graph()
G.add_nodes_from(range(len(df)))
# For each event, find neighboring events within the spatial radius and check if they are also close in time.
for i in range(len(df)):
    neighbors = tree.query_radius([coords[i]], r=radius)[0]

    for j in neighbors: # Don't compare the event with itself
        if i == j:
            continue

        if time_dist(i, j) <= TIME_THRESHOLD: # Apply time threshold --> add edge if both spatial and temporal conditions are met
            G.add_edge(i, j)

# -----------------------------
# CONNECTED COMPONENTS
# -----------------------------
components = list(nx.connected_components(G))
# Assign cluster labels based on connected components. Each component gets a unique label.
labels = np.full(len(df), -1)

for cid, comp in enumerate(components):
    for idx in comp:
        labels[idx] = cid

df["cluster"] = labels

In [21]:
# -----------------------------
# CLUSTER SUMMARY 
# -----------------------------

# only consider points that are part of a cluster (at least 3 points)
df_clean = df[df["cluster"] != -1]

summary = (
    df_clean
    .groupby("cluster")
    .size()
    .reset_index(name="fires_detected")   
    .sort_values("fires_detected", ascending=False)
)

# only Top 10 Cluster
top10 = summary.head(10)

# number of clusters with at least 3 fires
clusters_ge3 = (summary["fires_detected"] >= 3).sum()

print("Top 10 clusters by detected fires:\n")
print(top10.to_string(index=False))

print("\n Number of clusters with >= 3 fires:", clusters_ge3)

Top 10 clusters by detected fires:

 cluster  fires_detected
     148             179
     141             123
      59              94
     127              74
      55              59
     168              50
     144              47
      29              47
      16              46
      61              46

 Number of clusters with >= 3 fires: 213


# Cluster visualisation

In [22]:
import folium
import numpy as np
import pandas as pd
import matplotlib.cm as cm

# -----------------------------
# CLUSTER SIZE
# -----------------------------
cluster_sizes = df.groupby("cluster").size()

df["cluster_size"] = df["cluster"].map(cluster_sizes)

# -----------------------------
# COLOR SCALE (based on size)
# -----------------------------
max_size = cluster_sizes.max()
min_size = cluster_sizes.min()

colormap = cm.get_cmap("Reds")  

def get_color(size):
    if size is None or np.isnan(size):
        return "#000000"

    # normalisieren 0–1
    norm = (size - min_size) / (max_size - min_size + 1e-9)

    r, g, b, _ = colormap(norm)
    return f"#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}"


# -----------------------------
# MAP INIT
# -----------------------------
m = folium.Map(
    location=[df["lat"].mean(), df["lon"].mean()],
    zoom_start=5
)

# -----------------------------
# ADD POINTS
# -----------------------------
for _, row in df.iterrows():

    if row["cluster"] == -1:
        color = "#444444"  # noise = grau
    else:
        color = get_color(row["cluster_size"])

    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=3,
        color=color,
        fill=True,
        fill_opacity=0.8
    ).add_to(m)

from pathlib import Path
import webbrowser

out_file = Path("../output/viirs_clusters.html").resolve()
out_file.parent.mkdir(exist_ok=True)

m.save(out_file)

webbrowser.open(f"file://{out_file}")

C:\Users\Jan Krummenacher\AppData\Local\Temp\ipykernel_6388\3615335128.py:19: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colormap = cm.get_cmap("Reds")


True

In [23]:
import folium
from IPython.display import display
from IPython.display import HTML

# Display the map object. This will render the map.
HTML(m._repr_html_())

Map visualisation inside the notebook may be blocked by Visual Studio Code (VS does not trust this notebook and therefore does not visualize any map output). Visualisation with webbrowser works.

# Convex hull for each cluster

In [24]:
# Function to calculate convex hull area for a cluster
from scipy.spatial import ConvexHull, QhullError
import numpy as np

def safe_convex_hull(points):

    # ---------------------------------
    # remove duplicate coordinates
    # ---------------------------------
    points = np.unique(points, axis=0)

    # ---------------------------------
    # need at least 3 unique points
    # ---------------------------------
    if len(points) < 3:
        return None

    # ---------------------------------
    # detect collinear points
    # ---------------------------------
    if np.linalg.matrix_rank(points - points[0]) < 2:
        return None

    # ---------------------------------
    # compute hull
    # ---------------------------------
    try:
        hull = ConvexHull(points)
        return points[hull.vertices]

    except QhullError:
        return None

In [25]:
# Apply function to each cluster
hulls = {}
sizes = df.groupby("cluster").size()

for c in df["cluster"].unique():

    if c == -1:
        continue

    subset = df[df["cluster"] == c]
    points = subset[["lon", "lat"]].values

    hull_points = safe_convex_hull(points)

    if hull_points is not None:
        hulls[c] = hull_points

# We map our convex hulls 

In [26]:
import folium
import matplotlib.cm as cm
from pathlib import Path
import webbrowser

# Map center
m = folium.Map(
    location=[df["lat"].mean(), df["lon"].mean()],
    zoom_start=5
)

max_size = sizes.max()
colormap = cm.get_cmap("Reds")

def get_color(size):
    norm = size / max_size
    r, g, b, _ = colormap(norm)
    return f"#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}"

# -----------------------------
# DRAW HULLS
# -----------------------------
for cluster_id, hull in hulls.items():

    size = sizes[cluster_id]
    color = get_color(size)

    folium.Polygon(
        locations=[(lat, lon) for lon, lat in hull],
        color=color,
        weight=2,
        fill=True,
        fill_opacity=0.25,
        popup=f"Cluster {cluster_id} | {size} points"
    ).add_to(m)

# -----------------------------
# optional: points overlay
# -----------------------------
for _, row in df.iterrows():

    if row["cluster"] == -1:
        continue

    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=2,
        color="black",
        fill=True,
        fill_opacity=0.4
    ).add_to(m)

m # display map in notebook

out_file = Path("../output/convex_hull_clusters.html").resolve()
out_file.parent.mkdir(exist_ok=True)

m.save(out_file)

webbrowser.open(f"file://{out_file}")

C:\Users\Jan Krummenacher\AppData\Local\Temp\ipykernel_6388\2594176983.py:13: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colormap = cm.get_cmap("Reds")


True

In [27]:
import folium
from IPython.display import display
from IPython.display import HTML

# Display the map object. This will render the map.
HTML(m._repr_html_())

Map visualisation inside the notebook may be blocked by Visual Studio Code (VS does not trust this notebook and therefore does not visualize any map output). Visualisation with webbrowser works.